<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/usage-guides/4wings-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4Wings API

This guide provides detailed instructions on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to access the [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api), which is designed for generating reports and statistics on activities within specified regions.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be easily installed using pip:

In [1]:
# %pip install gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import os

import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Creating a Fishing Effort Report (`create_fishing_effort_report`)

Generates **AIS (Automatic Identification System) apparent fishing effort** reports to visualize fishing activity. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

In [38]:
fishing_effort_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="LOW",
    temporal_resolution="MONTHLY",
    group_by="FLAG",
    start_date="2022-01-01",
    end_date="2022-05-01",
    region={
        "dataset": "public-eez-areas",
        "id": "5690",
    },
)

### Access the report data as Pydantic models

In [39]:
fishing_effort_report_data = fishing_effort_report_result.data()

In [40]:
fishing_effort_report_item = fishing_effort_report_data[-1]

In [41]:
(
    fishing_effort_report_item.date,
    fishing_effort_report_item.flag,
    fishing_effort_report_item.hours,
    fishing_effort_report_item.vessel_ids,
    fishing_effort_report_item.lat,
    fishing_effort_report_item.lon,
)

('2022-03', 'RUS', 7.109166666666667, 3, 75.8, 44.0)

### Access the report data as a DataFrame

In [42]:
fishing_effort_report_df = fishing_effort_report_result.df()

In [43]:
fishing_effort_report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32271 entries, 0 to 32270
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     32271 non-null  object 
 1   detections               0 non-null      object 
 2   flag                     32271 non-null  object 
 3   gear_type                0 non-null      object 
 4   hours                    32271 non-null  float64
 5   vessel_ids               32271 non-null  int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_sign             

In [44]:
fishing_effort_report_df.head()

,date,detections,flag,gear_type,hours,vessel_ids,vessel_id,vessel_type,entry_timestamp,exit_timestamp,first_transmission_date,last_transmission_date,imo,mmsi,call_sign,dataset,report_dataset,ship_name,lat,lon
0,2022-02,None,RUS,None,58.298253,18,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,54.5,151.4
1,2022-02,None,RUS,None,53.717894,9,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,70.4,32.3
2,2022-02,None,RUS,None,1.083333,1,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,56.2,155.4
3,2022-01,None,RUS,None,5.364167,2,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,50.0,141.8
4,2022-03,None,RUS,None,2.393056,2,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,56.2,162.5


## Creating an AIS Presence Report (`create_ais_presence_report`)

Generates **AIS (Automatic Identification System) vessel presence** reports to visualize movement patterns of any vessel type. Please [learn more about AIS vessel presence here](https://globalfishingwatch.org/our-apis/documentation#ais-vessel-presence) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#ais-vessel-presence-caveats).

**Disclaimer:** AIS vessel presence is one of the largest datasets available. To prevent timeouts and ensure optimal performance, keep requests manageable: prefer simple, small regions and shorter time ranges (e.g., a few days).

In [45]:
ais_presence_report_result = await gfw_client.fourwings.create_ais_presence_report(
    spatial_resolution="LOW",
    temporal_resolution="MONTHLY",
    group_by="FLAG",
    start_date="2022-01-01",
    end_date="2022-05-01",
    region={
        "dataset": "public-eez-areas",
        "id": "5690",
    },
)

### Access the report data as Pydantic models

In [46]:
ais_presence_report_data = ais_presence_report_result.data()

In [47]:
ais_presence_report_item = ais_presence_report_data[-1]

In [49]:
(
    ais_presence_report_item.date,
    ais_presence_report_item.flag,
    ais_presence_report_item.hours,
    ais_presence_report_item.vessel_ids,
    ais_presence_report_item.lat,
    ais_presence_report_item.lon,
)

('2022-03', 'RUS', 1.0, 1, 52.1, 153.2)

### Access the report data as a DataFrame

In [50]:
ais_presence_report_df = ais_presence_report_result.df()

In [51]:
ais_presence_report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274333 entries, 0 to 274332
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   date                     274333 non-null  object 
 1   detections               0 non-null       object 
 2   flag                     274333 non-null  object 
 3   gear_type                0 non-null       object 
 4   hours                    274333 non-null  float64
 5   vessel_ids               274333 non-null  int64  
 6   vessel_id                0 non-null       object 
 7   vessel_type              0 non-null       object 
 8   entry_timestamp          0 non-null       object 
 9   exit_timestamp           0 non-null       object 
 10  first_transmission_date  0 non-null       object 
 11  last_transmission_date   0 non-null       object 
 12  imo                      0 non-null       object 
 13  mmsi                     0 non-null       object 
 14  call

In [52]:
ais_presence_report_df.head()

,date,detections,flag,gear_type,hours,vessel_ids,vessel_id,vessel_type,entry_timestamp,exit_timestamp,first_transmission_date,last_transmission_date,imo,mmsi,call_sign,dataset,report_dataset,ship_name,lat,lon
0,2022-04,None,PAN,None,3.0,2,None,None,None,None,None,None,None,None,None,None,public-global-presence:v3.0,None,47.8,154.1
1,2022-02,None,RUS,None,4.0,1,None,None,None,None,None,None,None,None,None,None,public-global-presence:v3.0,None,77.6,70.6
2,2022-02,None,RUS,None,1.0,1,None,None,None,None,None,None,None,None,None,None,public-global-presence:v3.0,None,75.0,159.2
3,2022-03,None,RUS,None,2.0,2,None,None,None,None,None,None,None,None,None,None,public-global-presence:v3.0,None,68.4,49.0
4,2022-02,None,PAN,None,1.0,1,None,None,None,None,None,None,None,None,None,None,public-global-presence:v3.0,None,50.0,157.0


## Creating a SAR Presence Report (`create_sar_presence_report`)

Generates **SAR (Synthetic-Aperture Radar) vessel detections** reports to identify vessels detected via radar, including non-broadcasting (possible `"dark"`) vessels. Please [learn more about SAR vessel detections here](https://globalfishingwatch.org/our-apis/documentation#sar-vessel-detections) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#sar-vessel-detections-data-caveats).

**Important:** **AIS vessel presence** shows where vessels **reported their positions** via the **Automatic Identification System (AIS)**. **SAR vessel detection** shows where **Synthetic Aperture Radar (SAR) satellites detected** vessels on the ocean surface, even if they **weren't transmitting AIS**.

In [19]:
sar_presence_report_result = await gfw_client.fourwings.create_sar_presence_report(
    spatial_resolution="LOW",
    temporal_resolution="MONTHLY",
    group_by="FLAG",
    start_date="2022-01-01",
    end_date="2022-05-01",
    region={
        "dataset": "public-eez-areas",
        "id": "5690",
    },
)

### Access the report data as Pydantic models

In [53]:
sar_presence_report_data = sar_presence_report_result.data()

In [54]:
sar_presence_report_item = sar_presence_report_data[-1]

In [55]:
(
    sar_presence_report_item.date,
    sar_presence_report_item.flag,
    sar_presence_report_item.detections,
    sar_presence_report_item.vessel_ids,
    sar_presence_report_item.lat,
    sar_presence_report_item.lon,
)

('2022-04', '', 1, 1, 46.6, 142.6)

### Access the report data as a DataFrame

In [56]:
sar_presence_report_df = sar_presence_report_result.df()

In [72]:
sar_presence_report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3995 entries, 0 to 3994
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     3995 non-null   object 
 1   detections               3995 non-null   int64  
 2   flag                     3995 non-null   object 
 3   gear_type                0 non-null      object 
 4   hours                    0 non-null      object 
 5   vessel_ids               3995 non-null   int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_sign               

In [58]:
sar_presence_report_df.head()

,date,detections,flag,gear_type,hours,vessel_ids,vessel_id,vessel_type,entry_timestamp,exit_timestamp,first_transmission_date,last_transmission_date,imo,mmsi,call_sign,dataset,report_dataset,ship_name,lat,lon
0,2022-02,1,LBR,None,None,1,None,None,None,None,None,None,None,None,None,None,public-global-sar-presence:v3.0,None,47.2,153.0
1,2022-02,1,BHS,None,None,1,None,None,None,None,None,None,None,None,None,None,public-global-sar-presence:v3.0,None,44.2,37.3
2,2022-02,1,,None,None,1,None,None,None,None,None,None,None,None,None,None,public-global-sar-presence:v3.0,None,46.3,142.4
3,2022-04,17,RUS,None,None,14,None,None,None,None,None,None,None,None,None,None,public-global-sar-presence:v3.0,None,45.5,36.7
4,2022-04,1,SGP,None,None,1,None,None,None,None,None,None,None,None,None,None,public-global-sar-presence:v3.0,None,46.3,151.9


## Creating a Generic Report (`create_report`)

Generates a report for any [supported datasets](https://globalfishingwatch.org/our-apis/documentation#supported-datasets), using fully customizable parameters. [Please check the data caveats here](https://globalfishingwatch.org/our-apis/documentation#data-caveat).

**Note:** AIS vessel presence (i.e., `"public-global-sar-presence:latest"` dataset) does **not** support `"GEARTYPE"` or `"FLAGANDGEARTYPE"` as `group_by` criteria.

In [26]:
report_result = await gfw_client.fourwings.create_report(
    spatial_resolution="LOW",
    temporal_resolution="MONTHLY",
    group_by="FLAG",
    datasets=[
        "public-global-fishing-effort:latest",
        "public-global-sar-presence:latest",
        "public-global-presence:latest",
    ],
    start_date="2022-01-01",
    end_date="2022-05-01",
    region={
        "dataset": "public-eez-areas",
        "id": "5690",
    },
)

### Access the report data as Pydantic models

In [67]:
report_data = report_result.data()

In [68]:
report_item = report_data[-1]

In [69]:
(
    report_item.date,
    report_item.flag,
    report_item.hours,
    report_item.vessel_ids,
    report_item.lat,
    report_item.lon,
)

('2022-03', 'RUS', 1.0, 1, 52.1, 153.2)

### Access the report data as a DataFrame

In [62]:
report_df = report_result.df()

In [66]:
report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310599 entries, 0 to 310598
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   date                     310599 non-null  object 
 1   detections               3995 non-null    float64
 2   flag                     310599 non-null  object 
 3   gear_type                0 non-null       object 
 4   hours                    306604 non-null  float64
 5   vessel_ids               310599 non-null  int64  
 6   vessel_id                0 non-null       object 
 7   vessel_type              0 non-null       object 
 8   entry_timestamp          0 non-null       object 
 9   exit_timestamp           0 non-null       object 
 10  first_transmission_date  0 non-null       object 
 11  last_transmission_date   0 non-null       object 
 12  imo                      0 non-null       object 
 13  mmsi                     0 non-null       object 
 14  call

In [64]:
report_df.head()

,date,detections,flag,gear_type,hours,vessel_ids,vessel_id,vessel_type,entry_timestamp,exit_timestamp,first_transmission_date,last_transmission_date,imo,mmsi,call_sign,dataset,report_dataset,ship_name,lat,lon
0,2022-03,NaN,RUS,None,3.416944,2,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,75.6,50.3
1,2022-02,NaN,RUS,None,2.144444,3,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,51.4,155.4
2,2022-03,NaN,RUS,None,5.436944,2,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,58.6,155.8
3,2022-01,NaN,RUS,None,4.435556,1,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,45.7,149.8
4,2022-03,NaN,RUS,None,0.362778,1,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v3.0,None,46.7,141.9
